# PrxteinMPNN Colab Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maraxen/prxteinmpnn/blob/main/examples/colab_inference_demo.ipynb)

## Installation

Install prxteinmpnn and dependencies. This example requires `prxteinmpnn==0.1.0a1` to be published to PyPI (run after release).

In [ ]:
# Install prxteinmpnn from PyPI
!pip install prxteinmpnn==0.1.0a1

## Imports

Import the public API from prxteinmpnn.

In [ ]:
from prxteinmpnn import (
    sample,
    score,
    SamplingSpecification,
    ScoringSpecification,
    RunSpecification,
    configure_multiprocessing,
)
import ipywidgets as widgets
from IPython.display import display, HTML
import json

print("Imports successful!")

## Interactive Inference Controls

Use the widgets below to configure your inference:

In [ ]:
# Create interactive widgets
seq_input = widgets.Text(
    value='MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLSGAEKAVQVKVKALPDAQFEVVHSLAKWKRQTLGQHDFSAGEGLYTHMKALRPDEDRLSPLHSVYVDQWDWERVMGDGERQFSTLKSTVEAIWAGIKATEAAVSEEFGLAPFLPDQIHFVHSQELLSRYPDLDAKGRERAIAKDLGAVFLVGIGGKLSDGHRHDVRAPDYDDWSTPSELGHAGLNGDILVWNPVLEDAFELSSMGIRVDADTLKHQLALTGDEDRLELEWHQALLRGEMPQTIGGGIGQSRLTMLLLQLPHIGQVQAGVWPAAVRESVPSLL',
    description='Protein Sequence:',
    layout=widgets.Layout(width='500px')
)

mode_dropdown = widgets.Dropdown(
    options=['score', 'sample'],
    value='score',
    description='Mode:'
)

temperature_slider = widgets.FloatSlider(
    value=0.5,
    min=0.1,
    max=1.0,
    step=0.05,
    description='Temperature:'
)

num_samples_spinner = widgets.IntSlider(
    value=1,
    min=1,
    max=10,
    step=1,
    description='Num Samples:',
)

run_button = widgets.Button(
    description='Run Inference',
    button_style='info',
    tooltip='Click to run the inference'
)

output_area = widgets.Output()

display(
    widgets.VBox([
        widgets.HTML('<h3>Configuration</h3>'),
        seq_input,
        mode_dropdown,
        temperature_slider,
        num_samples_spinner,
        run_button,
    ])
)

print("\nWidgets loaded. Configure your settings above and click 'Run Inference'.")

## Inference

Wire widget values into the public API and display results.

In [ ]:
def run_inference(button):
    """Execute inference based on current widget values."""
    output_area.clear_output(wait=True)
    
    with output_area:
        try:
            sequence = seq_input.value.strip()
            mode = mode_dropdown.value
            temperature = temperature_slider.value
            num_samples = num_samples_spinner.value
            
            # Validate sequence input
            if not sequence:
                print("Error: Please enter a protein sequence.")
                return
            
            if len(sequence) < 10:
                print(f"Warning: Sequence is very short ({len(sequence)} residues). Model works best on longer sequences.")
            
            print(f"\n=== PrxteinMPNN {mode.upper()} ===")
            print(f"Sequence length: {len(sequence)} residues")
            print(f"Temperature: {temperature}")
            
            if mode == 'score':
                # Score mode: evaluate sequence likelihood
                print(f"\nScoring sequence...")
                
                # Create a ScoringSpecification with the input sequence
                spec = ScoringSpecification(
                    sequence=sequence,
                    temperature=temperature,
                )
                
                # Run scoring
                result = score(spec)
                
                print(f"\nScore result:")
                print(f"  Input sequence: {sequence}")
                if hasattr(result, 'scores'):
                    print(f"  Scores: {result.scores}")
                else:
                    print(f"  Result: {result}")
            
            elif mode == 'sample':
                # Sample mode: generate new sequences
                print(f"\nSampling {num_samples} sequence(s)...")
                
                # Create a SamplingSpecification
                spec = SamplingSpecification(
                    sequence=sequence,
                    temperature=temperature,
                    num_samples=num_samples,
                )
                
                # Run sampling
                results = sample(spec)
                
                print(f"\nSample results:")
                print(f"  Input sequence: {sequence}")
                if isinstance(results, (list, tuple)):
                    for i, seq in enumerate(results):
                        print(f"  Sample {i+1}: {seq}")
                else:
                    print(f"  Generated: {results}")
            
            print("\n✓ Inference complete!")
        
        except Exception as e:
            print(f"Error during inference: {type(e).__name__}")
            print(f"Message: {str(e)}")
            print(f"\nTip: Ensure sequence is valid and model weights are accessible.")

# Attach the callback to the button
run_button.on_click(run_inference)

# Display the output area
display(widgets.HTML('<h3>Results</h3>'))
display(output_area)

## About PrxteinMPNN

PrxteinMPNN is a functional interface for ProteinMPNN, a deep learning model for protein design.

**Two main modes:**
- **Scoring**: Evaluate the likelihood of a protein sequence
- **Sampling**: Generate new protein sequences conditioned on a context sequence

**Key parameters:**
- **Temperature**: Controls diversity. Lower values (0.1) = more deterministic; higher values (1.0) = more random
- **Sequence**: The input protein sequence (one-letter amino acid codes)

**Learn more:**
- [GitHub Repository](https://github.com/maraxen/prxteinmpnn)
- [ProteinMPNN Paper](https://www.biorxiv.org/content/10.1101/2022.06.15.496402)